# A.6 找不到或文件不可靠時怎麼辦？


問書店地址，取回「貓在窗邊」為空；取回「書店地址尚未公佈」則有相關資料，卻仍沒有地址。前一種先補來源，後一種說明來源未提供。不要因回答可以寫得完整，就替兩種情況各編一個地址。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.retrieval import retrieve

query = "書店地址"
cases = [
    [{"id": "d1", "text": "貓在窗邊"}],
    [{"id": "d2", "text": "書店地址尚未公布"}],
]
for docs in cases:
    hits = retrieve(query, docs)
    if not hits:
        decision = "沒有取回相關來源，請補公告"
    elif "尚未公布" in hits[0]["text"]:
        decision = "來源未給地址，目前無法確定"
    else:
        decision = "需進一步核對來源與答案"
    print([d["id"] for d in hits], decision)

兩次依次是空清單與d2，決策分別為未取回來源、來源未給地址。這是程式的固定規則，沒有教模型誠實。把d2改成「書店地址是青街8號」，仍要核對店名、日期和答案，不能把非空當成可信。

兩份公告若一份青街8號、一份紅街9號，查看是否搬遷前後，或資料衝突無法確定。無法判斷時呈現衝突、請求較新來源；字詞高分不能決定哪份最新。

公告還可能夾帶「忽略問題」之類文字。資料中的地址可作證據，對助理下的新指令卻不是本次任務；這個邊界見[9.7](https://birdhackor.github.io/tiny-perceptron-vlm/9.7.html)。可回答題與資料不足題都保留，才能發現模型把UNKNOWN錯用在明明有答案的題目，而不是每題拒答就算誠實。

<details>
<summary>補充：舊RAG的缺資料回答</summary>

正式模型在無公告時12/12都生成`UNKNOWN`，只有別家公告時7/12如此；但正確公告已在眼前，仍有7/12回答`UNKNOWN`。它學到了某種缺資料回答格式，卻也在可回答題上錯用；這些數字不能當成一般誠實、安全或處理衝突的能力。這輪沒有另外評測文件中的惡意指令或自然語言版本衝突，不能替這些項目填成功。

</details>
